In [1]:
import os
import random
import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [2]:
import sys
import types
import numpy.core.numeric as real_numeric
core_pkg = types.ModuleType("numpy._core")
numeric_mod = types.ModuleType("numpy._core.numeric")
numeric_mod.__dict__.update(real_numeric.__dict__)
core_pkg.numeric = numeric_mod
sys.modules["numpy._core"] = core_pkg
sys.modules["numpy._core.numeric"] = numeric_mod

In [3]:
PATHS = {
    "default": r"C:\Users\MGA5500\Desktop\PROJECT\WW\Data",
    "cqr":     r"C:\Users\MGA5500\Desktop\PROJECT\WW\MSW codes Reimp",
}

ALL_EVAL_SEEDS = list(range(60, 70))
RF_TRAIN_SEEDS = list(range(70, 75))

NX       = 250
ALPHA    = 0.10
N_ITER   = 10
START_T  = 20
LIMIT_T  = 201
RNG_SEED = 1
Z_90 = 1.645
MAX_RF_SAMPLES = 200_000
TIME_STRIDE    = 1

SLICES = {
    "u": slice(0,      NX),
    "h": slice(NX,   2*NX),
    "r": slice(2*NX, 3*NX),
}
VARS = ["u", "h", "r"]

random.seed(RNG_SEED)
np.random.seed(RNG_SEED)

def safe_load_pickle(path):
    with open(path, "rb") as fh:
        try:
            return pickle.load(fh)
        except Exception:
            fh.seek(0)
            return pickle.load(fh, encoding="latin1")


def extract_analysis_at_t(obj, t):
    if isinstance(obj, dict) and "analysis" in obj:
        return obj["analysis"][t]
    return obj[t]


def ensure_state_2d(arr, three_nx):
    arr = np.asarray(arr)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    if arr.ndim == 2 and arr.shape[0] != three_nx and arr.shape[1] == three_nx:
        arr = arr.T
    return arr


def split_by_var(arr, slices=SLICES):
    return {v: np.asarray(arr[sl]) for v, sl in slices.items()}


def clip_nonnegative_interval(lo, hi):
    lo = np.maximum(np.asarray(lo, dtype=float), 0.0)
    hi = np.maximum(np.asarray(hi, dtype=float), 0.0)
    hi = np.maximum(hi, lo)
    return lo, hi


def interval_score(y, lo, hi, alpha):
    y  = np.asarray(y,  dtype=float)
    lo = np.asarray(lo, dtype=float)
    hi = np.asarray(hi, dtype=float)
    m  = np.isfinite(y) & np.isfinite(lo) & np.isfinite(hi)
    if not m.any():
        return np.nan, np.nan, np.nan, np.nan, np.nan

    y, lo, hi = y[m], lo[m], hi[m]
    width     = hi - lo
    under     = y < lo
    over      = y > hi
    score = width + (2.0/alpha)*(lo - y)*under + (2.0/alpha)*(y - hi)*over

    return (
        float(np.mean(score)),
        float(np.mean(~(under | over))),
        float(np.mean(under)),
        float(np.mean(over)),
        float(np.mean(width)),
    )


def interval_metrics_no_alpha(y, lo, hi):
    y  = np.asarray(y,  dtype=float)
    lo = np.asarray(lo, dtype=float)
    hi = np.asarray(hi, dtype=float)
    m  = np.isfinite(y) & np.isfinite(lo) & np.isfinite(hi)
    if not m.any():
        return np.nan, np.nan, np.nan, np.nan

    y, lo, hi = y[m], lo[m], hi[m]
    under = y < lo
    over  = y > hi
    return (
        float(np.mean(~(under | over))),
        float(np.mean(under)),
        float(np.mean(over)),
        float(np.mean(hi - lo)),
    )


def empirical_quantile(scores_1d, alpha):
    s = np.asarray(scores_1d).ravel()
    if s.size == 0:
        return 0.0
    q = float(np.ceil((s.size + 1) * (1 - alpha)) / s.size)
    return float(np.quantile(s, q, method="inverted_cdf"))

def get_eval_timesteps(base_path, seed, start_t=START_T, limit_T=None):
    nn = safe_load_pickle(os.path.join(base_path, str(seed), "NN_data.pkl"))
    T  = len(nn["analysis"]) if (isinstance(nn, dict) and "analysis" in nn) else len(nn)
    if limit_T is not None:
        T = min(T, int(limit_T))
    return list(range(start_t, T))


def load_truth(base_path, seed, t):
    truth_obj = safe_load_pickle(os.path.join(base_path, str(seed), "truth_data.pkl"))
    truth = np.asarray(truth_obj[t]).squeeze()
    if truth.ndim == 2:
        truth = truth.mean(axis=1)
    return truth


def load_default_arrays(seed, t):
    p     = os.path.join(PATHS["default"], str(seed))
    nn    = safe_load_pickle(os.path.join(p, "NN_data.pkl"))
    qpens = safe_load_pickle(os.path.join(p, "QPEns_data.pkl"))

    arr_nn = ensure_state_2d(extract_analysis_at_t(nn,    t), 3*NX)
    arr_qp = ensure_state_2d(extract_analysis_at_t(qpens, t), 3*NX)
    truth  = load_truth(PATHS["default"], seed, t)
    return arr_nn, arr_qp, truth


def load_default_means(seed, t):
    arr_nn, arr_qp, truth = load_default_arrays(seed, t)
    return (
        split_by_var(arr_nn.mean(axis=1)),
        split_by_var(arr_qp.mean(axis=1)),
        split_by_var(truth),
    )


def load_default_means_std(seed, t):
    arr_nn, arr_qp, truth = load_default_arrays(seed, t)
    nn_std = (arr_nn.std(axis=1, ddof=1) if arr_nn.shape[1] > 1
              else np.zeros(arr_nn.shape[0]))
    return (
        split_by_var(arr_nn.mean(axis=1)),
        split_by_var(nn_std),
        split_by_var(arr_qp.mean(axis=1)),
        split_by_var(truth),
    )


def load_default_full_ensembles(seed, t):
    arr_nn, arr_qp, truth = load_default_arrays(seed, t)
    nn_ens = {v: arr_nn[sl, :].T for v, sl in SLICES.items()}
    qp_ens = {v: arr_qp[sl, :].T for v, sl in SLICES.items()}
    return nn_ens, qp_ens, split_by_var(truth)


def load_cqr_data(seed, t):
    p     = os.path.join(PATHS["cqr"], str(seed))
    nn    = safe_load_pickle(os.path.join(p, "NN_data.pkl"))
    qpens = safe_load_pickle(os.path.join(p, "QPEns_data.pkl"))

    q_arr  = ensure_state_2d(extract_analysis_at_t(qpens, t), 3*NX)  
    lo_all = np.asarray(nn["lower"][t]).T   
    hi_all = np.asarray(nn["upper"][t]).T   
    truth  = load_truth(PATHS["cqr"], seed, t)
    return q_arr, lo_all, hi_all, truth


def make_row(method, eval_mode, iteration_id, t, seed, target, var,
             AISL, cov, ml, mh, w):
    return {
        "method": method,   "eval_mode": eval_mode,
        "iter":   iteration_id, "time": t, "seed": seed,
        "target": target,   "var": var,
        "AISL":   AISL,     "coverage":   cov,
        "miss_low": ml,     "miss_high":  mh,
        "mean_width": w,
    }

def _scp_band(t, calib_seeds, mode):
    nn_vals = {v: [] for v in VARS}
    qp_vals = {v: [] for v in VARS}

    for seed in calib_seeds:
        try:
            if mode == "members":
                arr_nn, arr_qp, _ = load_default_arrays(seed, t)
                for v, sl in SLICES.items():
                    nn_vals[v].append(arr_nn[sl, :])
                    qp_vals[v].append(arr_qp[sl, :])
            else:
                nn_mean, qp_mean, _ = load_default_means(seed, t)
                for v in VARS:
                    nn_vals[v].append(nn_mean[v])
                    qp_vals[v].append(qp_mean[v])
        except Exception as e:
            print(f"  SCP calibration [{mode}, seed={seed}, t={t}]: {e}")

    band = {}
    for v in VARS:
        if not nn_vals[v]:
            band[v] = np.nan
            continue
        scores = np.abs(np.stack(qp_vals[v]) - np.stack(nn_vals[v])).ravel()
        N      = scores.size
        qprob  = float(np.ceil((N + 1) * (1 - ALPHA)) / N)
        band[v] = float(np.quantile(scores, qprob, method="inverted_cdf"))

    return band


def evaluate_scp(t, test_seeds, calib_seeds, iteration_id, mode):
    rows = []
    band = _scp_band(t, calib_seeds, mode)

    for seed in test_seeds:
        try:
            if mode == "members":
                nn_ens, qp_ens, truth = load_default_full_ensembles(seed, t)
                for v in VARS:
                    b = band.get(v, np.nan)
                    if not np.isfinite(b):
                        continue
                    nnv = np.asarray(nn_ens[v])   
                    qv  = np.asarray(qp_ens[v])  
                    lo, hi = clip_nonnegative_interval(nnv - b, nnv + b)

                    AISL, cov, ml, mh, w = interval_score(
                        qv.ravel(), lo.ravel(), hi.ravel(), ALPHA)
                    rows.append(make_row("SCP", "members", iteration_id, t, seed,
                                         "QPEns_members", v, AISL, cov, ml, mh, w))

                    truth_flat = np.tile(truth[v], nnv.shape[0])
                    AISL, cov, ml, mh, w = interval_score(
                        truth_flat, lo.ravel(), hi.ravel(), ALPHA)
                    rows.append(make_row("SCP", "members", iteration_id, t, seed,
                                         "Truth", v, AISL, cov, ml, mh, w))
            else:
                nn_mean, qp_mean, truth = load_default_means(seed, t)
                for v in VARS:
                    b = band.get(v, np.nan)
                    if not np.isfinite(b):
                        continue
                    center = np.asarray(nn_mean[v])
                    lo, hi = clip_nonnegative_interval(center - b, center + b)

                    AISL, cov, ml, mh, w = interval_score(
                        np.asarray(qp_mean[v]), lo, hi, ALPHA)
                    rows.append(make_row("SCP", "mean", iteration_id, t, seed,
                                         "QPEns_mean", v, AISL, cov, ml, mh, w))

                    AISL, cov, ml, mh, w = interval_score(
                        np.asarray(truth[v]), lo, hi, ALPHA)
                    rows.append(make_row("SCP", "mean", iteration_id, t, seed,
                                         "Truth", v, AISL, cov, ml, mh, w))
        except Exception as e:
            print(f"  SCP evaluation [{mode}, seed={seed}, t={t}]: {e}")

    return pd.DataFrame(rows)

def train_rf_sigma(times):
    rf_models = {}
    print("Training RF sigma models for NCP-members ...")

    for v in VARS:
        X_list, y_list = [], []
        for seed in RF_TRAIN_SEEDS:
            for t in times[::TIME_STRIDE]:
                try:
                    nn_ens, qp_ens, _ = load_default_full_ensembles(seed, t)
                    nnv = np.asarray(nn_ens[v])
                    qv  = np.asarray(qp_ens[v])
                    X_list.append(np.stack([nnv, qv], axis=-1).reshape(-1, 2))
                    y_list.append(np.abs(qv - nnv).ravel())
                except Exception as e:
                    print(f"  RF training [var={v}, seed={seed}, t={t}]: {e}")

        if not X_list:
            rf_models[v] = None
            continue

        X = np.concatenate(X_list)
        y = np.concatenate(y_list)

        if len(y) > MAX_RF_SAMPLES:
            rng = np.random.default_rng(RNG_SEED + 1)
            idx = rng.choice(len(y), size=MAX_RF_SAMPLES, replace=False)
            X, y = X[idx], y[idx]

        rf = RandomForestRegressor(n_estimators=100, max_depth=20,
                                   random_state=RNG_SEED, n_jobs=-1)
        rf.fit(X, y)
        rf_models[v] = rf
        print(f"  var={v}: n_train={len(y)}, "
              f"MAE={mean_absolute_error(y, rf.predict(X)):.4f}")

    return rf_models


def _ncp_members_qscore(t, calib_seeds, rf_models):
    cpscore = {}
    for v in VARS:
        nn_list, q_list = [], []
        for seed in calib_seeds:
            try:
                nn_ens, qp_ens, _ = load_default_full_ensembles(seed, t)
                nn_list.append(np.asarray(nn_ens[v]))
                q_list.append(np.asarray(qp_ens[v]))
            except Exception as e:
                print(f"  NCP-members calibration [var={v}, seed={seed}, t={t}]: {e}")

        if not nn_list:
            cpscore[v] = np.nan
            continue

        nn_s = np.stack(nn_list)
        q_s  = np.stack(q_list)
        res  = np.abs(q_s - nn_s)

        if v in ("u", "h") and rf_models.get(v) is not None:
            X     = np.stack([nn_s, q_s], axis=-1).reshape(-1, 2)
            sigma = rf_models[v].predict(X).reshape(res.shape)
            valid  = sigma != 0
            scores = np.where(valid, res / sigma,
                     np.where(res == 0, 0.0, np.nan))
        else:
            scores = res

        s = scores.ravel()
        s = s[np.isfinite(s)]
        if s.size == 0:
            cpscore[v] = np.nan
            continue

        N     = s.size
        qprob = float(np.ceil((N + 1) * (1 - ALPHA)) / N)
        cpscore[v] = float(np.quantile(s, qprob, method="inverted_cdf"))

    return cpscore


def evaluate_ncp_members(t, test_seeds, calib_seeds, rf_models, iteration_id):
    rows    = []
    qscores = _ncp_members_qscore(t, calib_seeds, rf_models)

    for seed in test_seeds:
        try:
            nn_ens, qp_ens, truth = load_default_full_ensembles(seed, t)
            for v in VARS:
                q_s = qscores.get(v, np.nan)
                if not np.isfinite(q_s):
                    continue

                nnv = np.asarray(nn_ens[v]) 
                qv  = np.asarray(qp_ens[v])  

                if v in ("u", "h") and rf_models.get(v) is not None:
                    X     = np.stack([nnv, qv], axis=-1).reshape(-1, 2)
                    sigma = rf_models[v].predict(X).reshape(nnv.shape)
                    band  = np.where(np.isfinite(sigma), q_s * sigma, np.nan)
                else:
                    band = np.full_like(nnv, q_s, dtype=float)

                lo, hi = clip_nonnegative_interval(nnv - band, nnv + band)

                AISL, cov, ml, mh, w = interval_score(
                    qv.ravel(), lo.ravel(), hi.ravel(), ALPHA)
                rows.append(make_row("NCP", "members", iteration_id, t, seed,
                                     "QPEns_members", v, AISL, cov, ml, mh, w))

                truth_flat = np.tile(truth[v], nnv.shape[0])
                AISL, cov, ml, mh, w = interval_score(
                    truth_flat, lo.ravel(), hi.ravel(), ALPHA)
                rows.append(make_row("NCP", "members", iteration_id, t, seed,
                                     "Truth", v, AISL, cov, ml, mh, w))
        except Exception as e:
            print(f"  NCP-members evaluation [seed={seed}, t={t}]: {e}")

    return pd.DataFrame(rows)

def _ncp_mean_qscore(t, calib_seeds):
    cpscore = {}
    for v in VARS:
        nn_list, q_list, sigma_list = [], [], []
        for seed in calib_seeds:
            try:
                nn_mean, nn_std, qp_mean, _ = load_default_means_std(seed, t)
                nn_list.append(np.asarray(nn_mean[v]))
                q_list.append(np.asarray(qp_mean[v]))
                sigma_list.append(np.asarray(nn_std[v]))
            except Exception as e:
                print(f"  NCP-mean calibration [var={v}, seed={seed}, t={t}]: {e}")

        if not nn_list:
            cpscore[v] = np.nan
            continue

        nn_s    = np.stack(nn_list)
        q_s     = np.stack(q_list)
        sigma_s = np.stack(sigma_list)
        res     = np.abs(q_s - nn_s)

        if v in ("u", "h"):
            valid  = sigma_s != 0
            scores = np.where(valid, res / sigma_s,
                     np.where(res == 0, 0.0, np.nan))
        else:
            scores = res

        s = scores.ravel()
        s = s[np.isfinite(s)]
        if s.size == 0:
            cpscore[v] = np.nan
            continue

        N     = s.size
        qprob = float(np.ceil((N + 1) * (1 - ALPHA)) / N)
        cpscore[v] = float(np.quantile(s, qprob, method="inverted_cdf"))

    return cpscore


def evaluate_ncp_mean(t, test_seeds, calib_seeds, iteration_id):
    rows    = []
    qscores = _ncp_mean_qscore(t, calib_seeds)

    for seed in test_seeds:
        try:
            nn_mean, nn_std, qp_mean, truth = load_default_means_std(seed, t)
            for v in VARS:
                q_s = qscores.get(v, np.nan)
                if not np.isfinite(q_s):
                    continue

                center = np.asarray(nn_mean[v])
                sigma  = np.asarray(nn_std[v])

                if v in ("u", "h"):
                    band = np.where(sigma != 0, q_s * sigma, 0.0)
                else:
                    band = np.full_like(center, q_s, dtype=float)

                lo, hi = clip_nonnegative_interval(center - band, center + band)

                AISL, cov, ml, mh, w = interval_score(
                    np.asarray(qp_mean[v]), lo, hi, ALPHA)
                rows.append(make_row("NCP", "mean", iteration_id, t, seed,
                                     "QPEns_mean", v, AISL, cov, ml, mh, w))

                AISL, cov, ml, mh, w = interval_score(
                    np.asarray(truth[v]), lo, hi, ALPHA)
                rows.append(make_row("NCP", "mean", iteration_id, t, seed,
                                     "Truth", v, AISL, cov, ml, mh, w))
        except Exception as e:
            print(f"  NCP-mean evaluation [seed={seed}, t={t}]: {e}")

    return pd.DataFrame(rows)

def _cqr_deltas(t, calib_seeds, mode):
    scores = {v: [] for v in VARS}

    for seed in calib_seeds:
        try:
            q_arr, lo_all, hi_all, _ = load_cqr_data(seed, t)

            if mode == "members":
                for v in VARS:
                    sl   = SLICES[v]
                    lo_v = lo_all[:, sl]    
                    hi_v = hi_all[:, sl]    
                    y_v  = q_arr[sl, :].T   
                    s    = np.maximum(np.maximum(lo_v - y_v, y_v - hi_v), 0.0)
                    scores[v].append(s.ravel())
            else:
                lo_mean = lo_all.mean(axis=0)   
                hi_mean = hi_all.mean(axis=0)   
                q_mean  = q_arr.mean(axis=1)  
                for v in VARS:
                    sl = SLICES[v]
                    s  = np.maximum(
                             np.maximum(lo_mean[sl] - q_mean[sl],
                                        q_mean[sl] - hi_mean[sl]),
                             0.0)
                    scores[v].append(s)
        except Exception as e:
            print(f"  CQR calibration [{mode}, seed={seed}, t={t}]: {e}")

    return {v: (empirical_quantile(np.concatenate(scores[v]), ALPHA)
                if scores[v] else np.nan)
            for v in VARS}


def evaluate_cqr(t, test_seeds, calib_seeds, iteration_id, mode):
    rows   = []
    deltas = _cqr_deltas(t, calib_seeds, mode)

    for seed in test_seeds:
        try:
            q_arr, lo_all, hi_all, truth_all = load_cqr_data(seed, t)
            truth = split_by_var(truth_all)

            if mode == "members":
                lo = lo_all.copy()   
                hi = hi_all.copy()   
                for v in VARS:
                    d = deltas.get(v, np.nan)
                    if np.isfinite(d):
                        lo[:, SLICES[v]] -= d
                        hi[:, SLICES[v]] += d

                for v in VARS:
                    d = deltas.get(v, np.nan)
                    if not np.isfinite(d):
                        continue
                    sl   = SLICES[v]
                    lo_v = lo[:, sl]      
                    hi_v = hi[:, sl]     
                    y_v  = q_arr[sl, :].T 

                    AISL, cov, ml, mh, w = interval_score(
                        y_v.ravel(), lo_v.ravel(), hi_v.ravel(), ALPHA)
                    rows.append(make_row("CQR", "members", iteration_id, t, seed,
                                         "QPEns_members", v, AISL, cov, ml, mh, w))

                    truth_flat = np.tile(truth[v], lo_v.shape[0])
                    AISL, cov, ml, mh, w = interval_score(
                        truth_flat, lo_v.ravel(), hi_v.ravel(), ALPHA)
                    rows.append(make_row("CQR", "members", iteration_id, t, seed,
                                         "Truth", v, AISL, cov, ml, mh, w))
            else:
                lo_mean = lo_all.mean(axis=0)  
                hi_mean = hi_all.mean(axis=0)   
                q_mean  = q_arr.mean(axis=1)   

                for v in VARS:
                    d = deltas.get(v, np.nan)
                    if not np.isfinite(d):
                        continue
                    sl   = SLICES[v]
                    lo_v = lo_mean[sl] - d
                    hi_v = hi_mean[sl] + d

                    AISL, cov, ml, mh, w = interval_score(q_mean[sl], lo_v, hi_v, ALPHA)
                    rows.append(make_row("CQR", "mean", iteration_id, t, seed,
                                         "QPEns_mean", v, AISL, cov, ml, mh, w))

                    AISL, cov, ml, mh, w = interval_score(
                        np.asarray(truth[v]), lo_v, hi_v, ALPHA)
                    rows.append(make_row("CQR", "mean", iteration_id, t, seed,
                                         "Truth", v, AISL, cov, ml, mh, w))
        except Exception as e:
            print(f"  CQR evaluation [{mode}, seed={seed}, t={t}]: {e}")

    return pd.DataFrame(rows)

def evaluate_ensemble_spread(t, test_seeds, iteration_id):
    rows = []
    for seed in test_seeds:
        try:
            arr_nn, arr_qp, truth_all = load_default_arrays(seed, t)
            nn_min  = split_by_var(arr_nn.min(axis=1))
            nn_max  = split_by_var(arr_nn.max(axis=1))
            qp_mean = split_by_var(arr_qp.mean(axis=1))
            truth   = split_by_var(truth_all)

            for v in VARS:
                lo, hi = clip_nonnegative_interval(nn_min[v], nn_max[v])

                cov, ml, mh, w = interval_metrics_no_alpha(qp_mean[v], lo, hi)
                rows.append(make_row("EnsembleSpread", "spread", iteration_id, t, seed,
                                     "QPEns_mean", v, np.nan, cov, ml, mh, w))

                cov, ml, mh, w = interval_metrics_no_alpha(truth[v], lo, hi)
                rows.append(make_row("EnsembleSpread", "spread", iteration_id, t, seed,
                                     "Truth", v, np.nan, cov, ml, mh, w))
        except Exception as e:
            print(f"  EnsembleSpread evaluation [seed={seed}, t={t}]: {e}")

    return pd.DataFrame(rows)


def evaluate_std_baseline(t, test_seeds, iteration_id):
    rows = []
    for seed in test_seeds:
        try:
            nn_mean, nn_std, qp_mean, truth = load_default_means_std(seed, t)
            for v in VARS:
                center = np.asarray(nn_mean[v])
                sigma  = np.asarray(nn_std[v])
                lo, hi = clip_nonnegative_interval(center - Z_90 * sigma,
                                                   center + Z_90 * sigma)

                AISL, cov, ml, mh, w = interval_score(
                    np.asarray(qp_mean[v]), lo, hi, ALPHA)
                rows.append(make_row("STD", "mean", iteration_id, t, seed,
                                     "QPEns_mean", v, AISL, cov, ml, mh, w))

                AISL, cov, ml, mh, w = interval_score(
                    np.asarray(truth[v]), lo, hi, ALPHA)
                rows.append(make_row("STD", "mean", iteration_id, t, seed,
                                     "Truth", v, AISL, cov, ml, mh, w))
        except Exception as e:
            print(f"  STD evaluation [seed={seed}, t={t}]: {e}")

    return pd.DataFrame(rows)

def summarize_results(df):
    if df.empty:
        return pd.DataFrame(columns=[
            "method", "eval_mode", "target", "var",
            "AISL_mean", "AISL_std",
            "coverage_mean", "coverage_std",
            "width_mean", "width_std",
            "miss_low_mean", "miss_low_std",
            "miss_high_mean", "miss_high_std",
        ])

    return (
        df.groupby(["method", "eval_mode", "target", "var"])
          .agg(
              AISL_mean     =("AISL",       "mean"),
              AISL_std      =("AISL",       "std"),
              coverage_mean =("coverage",   "mean"),
              coverage_std  =("coverage",   "std"),
              width_mean    =("mean_width", "mean"),
              width_std     =("mean_width", "std"),
              miss_low_mean =("miss_low",   "mean"),
              miss_low_std  =("miss_low",   "std"),
              miss_high_mean=("miss_high",  "mean"),
              miss_high_std =("miss_high",  "std"),
          )
          .reset_index()
    )


def metric_table(summary, method, eval_mode, mean_col, std_col, target, decimals=4):
    sub = (
        summary[(summary["method"]    == method) &
                (summary["eval_mode"] == eval_mode) &
                (summary["target"]    == target)]
        .set_index("var").reindex(["u", "h", "r"])
    )
    col = f"{method} | {eval_mode} | {target}"
    out = pd.DataFrame(index=["u", "h", "r"], columns=[col], dtype=object)
    for v in ["u", "h", "r"]:
        m = sub.loc[v, mean_col] if (v in sub.index and mean_col in sub.columns) else np.nan
        s = sub.loc[v, std_col]  if (v in sub.index and std_col  in sub.columns) else np.nan
        out.loc[v, col] = (f"{m:.{decimals}f} ± {s:.{decimals}f}"
                           if np.isfinite(m) and np.isfinite(s) else "NaN")
    return out

def make_splits(all_seeds, n_iter, rng_seed=123):
    rng    = random.Random(rng_seed)
    splits = []
    for i in range(n_iter):
        seeds = all_seeds.copy()
        rng.shuffle(seeds)
        mid   = len(seeds) // 2
        calib = seeds[:mid]
        test  = seeds[mid:]
        if not test:
            test  = [calib[-1]]
            calib = calib[:-1]
        splits.append({"iter": i, "calib_seeds": calib, "test_seeds": test})
    return splits

def run_evaluation():
    times_default = get_eval_timesteps(PATHS["default"], ALL_EVAL_SEEDS[0])
    times_cqr     = get_eval_timesteps(PATHS["cqr"],     ALL_EVAL_SEEDS[0])
    times         = sorted(set(times_default) & set(times_cqr))

    if not times:
        print("No common timesteps found between default and CQR paths.")
        return pd.DataFrame(), pd.DataFrame()

    print(f"Evaluating {len(times)} timesteps: t={times[0]} to t={times[-1]}")

    splits    = make_splits(ALL_EVAL_SEEDS, N_ITER, rng_seed=RNG_SEED)
    rf_models = train_rf_sigma(times)

    frames = []

    for sp in splits:
        i_id  = sp["iter"]
        calib = sp["calib_seeds"]
        test  = sp["test_seeds"]

        print(f"\n--- Iteration {i_id+1}/{N_ITER} | "
              f"calib={calib} | test={test} ---")

        for t in times:
            dfs = [
                evaluate_scp(t, test, calib, i_id, mode="members"),
                evaluate_scp(t, test, calib, i_id, mode="mean"),
                evaluate_ncp_members(t, test, calib, rf_models, i_id),
                evaluate_ncp_mean(t, test, calib, i_id),
                evaluate_cqr(t, test, calib, i_id, mode="members"),
                evaluate_cqr(t, test, calib, i_id, mode="mean"),
                evaluate_ensemble_spread(t, test, i_id),
                evaluate_std_baseline(t, test, i_id),
            ]
            frames.extend(df for df in dfs if df is not None and not df.empty)

    if not frames:
        return pd.DataFrame(), pd.DataFrame()

    df_all  = pd.concat(frames, ignore_index=True)
    summary = summarize_results(df_all)
    return df_all, summary

df_all, summary = run_evaluation()

df_all  = df_all.round(4)
summary = summary.round(4)

print("\n========== Summary ==========")
print(summary.sort_values(["method", "eval_mode", "target", "var"]).to_string())

out_dir = os.path.join(PATHS["default"], "AISL_results")
os.makedirs(out_dir, exist_ok=True)
df_all.to_csv( os.path.join(out_dir, "detailed_results.csv"), index=False)
summary.to_csv(os.path.join(out_dir, "summary_results.csv"),  index=False)
print(f"\nResults saved to: {out_dir}")

# Printing
views = [
    ("SCP",            "members", "QPEns_members"),
    ("SCP",            "members", "Truth"),
    ("SCP",            "mean",    "QPEns_mean"),
    ("SCP",            "mean",    "Truth"),
    ("NCP",            "members", "QPEns_members"),
    ("NCP",            "members", "Truth"),
    ("NCP",            "mean",    "QPEns_mean"),
    ("NCP",            "mean",    "Truth"),
    ("CQR",            "members", "QPEns_members"),
    ("CQR",            "members", "Truth"),
    ("CQR",            "mean",    "QPEns_mean"),
    ("CQR",            "mean",    "Truth"),
    ("STD",            "mean",    "QPEns_mean"),
    ("STD",            "mean",    "Truth"),
    ("EnsembleSpread", "spread",  "QPEns_mean"),
    ("EnsembleSpread", "spread",  "Truth"),
]

METRIC_COLS = [
    ("AISL",      "AISL_mean",      "AISL_std"),
    ("Coverage",  "coverage_mean",  "coverage_std"),
    ("Width",     "width_mean",     "width_std"),
    ("Miss Low",  "miss_low_mean",  "miss_low_std"),
    ("Miss High", "miss_high_mean", "miss_high_std"),
]

for method, eval_mode, target in views:
    sub = summary[(summary["method"]    == method) &
                  (summary["eval_mode"] == eval_mode) &
                  (summary["target"]    == target)]
    if sub.empty:
        continue

    for metric_name, mc, sc in METRIC_COLS:
        if metric_name == "AISL" and method == "EnsembleSpread":
            continue  
        print(f"\n{metric_name} -- {method} | {eval_mode} | {target}")
        print(metric_table(summary, method, eval_mode, mc, sc, target))


Evaluating 181 timesteps: t=20 to t=200
Training RF sigma models for NCP-members ...
  var=u: n_train=200000, MAE=0.0001
  var=h: n_train=200000, MAE=0.0004
  var=r: n_train=200000, MAE=0.0000

--- Iteration 1/10 | calib=[66, 68, 69, 67, 65] | test=[63, 60, 64, 61, 62] ---

--- Iteration 2/10 | calib=[64, 68, 62, 66, 65] | test=[69, 60, 67, 61, 63] ---

--- Iteration 3/10 | calib=[67, 68, 66, 69, 65] | test=[60, 62, 61, 63, 64] ---

--- Iteration 4/10 | calib=[67, 62, 68, 61, 64] | test=[60, 65, 69, 63, 66] ---

--- Iteration 5/10 | calib=[62, 60, 64, 69, 66] | test=[67, 61, 65, 63, 68] ---

--- Iteration 6/10 | calib=[63, 69, 64, 60, 62] | test=[65, 67, 61, 68, 66] ---

--- Iteration 7/10 | calib=[60, 61, 65, 67, 64] | test=[69, 62, 63, 66, 68] ---

--- Iteration 8/10 | calib=[66, 68, 62, 64, 63] | test=[65, 61, 67, 60, 69] ---

--- Iteration 9/10 | calib=[62, 61, 67, 66, 64] | test=[63, 60, 69, 68, 65] ---

--- Iteration 10/10 | calib=[61, 69, 64, 66, 62] | test=[68, 63, 60, 67, 65] 


Results saved to: C:\Users\MGA5500\Desktop\PROJECT\WW\Data\AISL_results

AISL -- SCP | members | QPEns_members
  SCP | members | QPEns_members
u               0.0088 ± 0.0014
h               0.0946 ± 0.0278
r               0.0126 ± 0.0052

Coverage -- SCP | members | QPEns_members
  SCP | members | QPEns_members
u               0.8989 ± 0.0302
h               0.8982 ± 0.0388
r               0.8964 ± 0.0385

Width -- SCP | members | QPEns_members
  SCP | members | QPEns_members
u               0.0046 ± 0.0004
h               0.0347 ± 0.0123
r               0.0015 ± 0.0007

Miss Low -- SCP | members | QPEns_members
  SCP | members | QPEns_members
u               0.0494 ± 0.0154
h               0.0522 ± 0.0231
r               0.0571 ± 0.0239

Miss High -- SCP | members | QPEns_members
  SCP | members | QPEns_members
u               0.0517 ± 0.0184
h               0.0495 ± 0.0196
r               0.0465 ± 0.0193

AISL -- SCP | members | Truth
  SCP | members | Truth
u       0.0116 ± 0.0015